# NYC Taxi | Silver analysis

## Goal
Compare eligible pickup groups and inspect the pickup-hour distribution for January 2025.

## Setup
Run notebook 02 first. This notebook reads `workspace.default.nyc_taxi_silver_2025_01` and creates no Gold table. Recorded outputs from unchanged queries are retained.

**Population:** borough and Queens comparisons use measurement-eligible records; the hourly volume query includes all January records.

## Steps

### Compare pickup boroughs

In [ ]:
%sql
SELECT
    pickup_borough,
    COUNT(*) AS secilen_yolculuk_sayisi,
    ROUND(AVG(trip_distance), 2) AS ortalama_mesafe_mil,
    ROUND(AVG(sure_dakika), 2) AS ortalama_sure_dakika
FROM workspace.default.nyc_taxi_silver_2025_01
WHERE olcum_analizine_uygun = true
GROUP BY pickup_borough
ORDER BY secilen_yolculuk_sayisi DESC;

pickup_borough,secilen_yolculuk_sayisi,ortalama_mesafe_mil,ortalama_sure_dakika
Manhattan,3013431,2.28,12.78
Queens,283713,12.02,31.45
Brooklyn,61268,5.4,26.8
Bronx,13604,7.46,31.41
Unknown,7697,3.38,14.83
N/A,573,8.08,16.17
Staten Island,230,9.04,24.29
EWR,76,3.93,8.92


### Inspect the top Queens pickup zones

In [ ]:
%sql
SELECT
    pickup_zone,
    COUNT(*) AS yolculuk_sayisi,
    ROUND(AVG(trip_distance), 2) AS ortalama_mesafe_mil,
    ROUND(AVG(sure_dakika), 2) AS ortalama_sure_dakika
FROM workspace.default.nyc_taxi_silver_2025_01
WHERE olcum_analizine_uygun = true
  AND pickup_borough = 'Queens'
GROUP BY pickup_zone
ORDER BY yolculuk_sayisi DESC
LIMIT 10;

pickup_zone,yolculuk_sayisi,ortalama_mesafe_mil,ortalama_sure_dakika
JFK Airport,142609,15.42,37.55
LaGuardia Airport,88471,9.6,26.26
East Elmhurst,9662,9.29,25.79
Sunnyside,3625,5.2,19.99
Astoria,2954,4.58,19.11
Long Island City/Hunters Point,2752,5.43,19.69
South Ozone Park,2148,5.39,18.68
Jamaica,2146,7.02,23.02
Jackson Heights,2042,4.8,17.84
Richmond Hill,1774,6.09,23.83


### Compare Queens airports with other Queens zones

In [ ]:
%sql
SELECT
    CASE
        WHEN pickup_zone IN ('JFK Airport', 'LaGuardia Airport')
            THEN 'Havalimanı'
        ELSE 'Queens diğer bölgeler'
    END AS binis_grubu,
    COUNT(*) AS yolculuk_sayisi,
    ROUND(AVG(trip_distance), 2) AS ortalama_mesafe_mil,
    ROUND(AVG(sure_dakika), 2) AS ortalama_sure_dakika
FROM workspace.default.nyc_taxi_silver_2025_01
WHERE olcum_analizine_uygun = true
  AND pickup_borough = 'Queens'
GROUP BY 1;

binis_grubu,yolculuk_sayisi,ortalama_mesafe_mil,ortalama_sure_dakika
Havalimanı,231080,13.19,33.23
Queens diğer bölgeler,52633,6.88,23.66


### Count all January records by pickup hour

In [ ]:
%sql
SELECT
    HOUR(tpep_pickup_datetime) AS binis_saati,
    COUNT(*) AS yolculuk_kaydi_sayisi
FROM workspace.default.nyc_taxi_silver_2025_01
GROUP BY binis_saati
ORDER BY binis_saati;

binis_saati,yolculuk_kaydi_sayisi
0,93416
1,64484
2,43929
3,28492
4,20033
5,22551
6,50026
7,102581
8,141305
9,142877


## Checks and takeaways
Recorded outputs show:
- Manhattan: 3,013,431 measurement-eligible pickup records; mean distance 2.28 miles and mean duration 12.78 minutes.
- Queens: 283,713 eligible pickup records; mean distance 12.02 miles and mean duration 31.45 minutes.
- JFK and LaGuardia together: 231,080 of the eligible Queens pickup records.
- Highest month-wide pickup-hour total: 18:00–18:59, with 267,951 records.
- Lowest month-wide pickup-hour total: 04:00–04:59, with 20,033 records.

These are descriptive associations. Airport mix may help interpret the Queens averages; this notebook does not establish a causal effect or measure traffic congestion.

## Next steps
Gold creation and reconciliation are owned only by notebook 04.